# 03b. 8월5일 피크 LST 구름 갭필

폭염 지수의 LST 축 = 8/5 피크. 그러나 8/5은 구름 55%(특히 북부 창릉·왕숙에 집중).
**표면유형(NDVI·blockType) + 위치(UTM 좌표)**로 결측 격자를 예측해 full-coverage 피크 지도를 만든다.

설계 근거:
- 6월/합성 LST는 예측변수로 **제외**(계절 역전 → 그린필드 오예측).
- 위치(cy=북향)로 위도·서울 북쪽 효과 반영. **지형(DEM)은 미수집 → v1 미반영(개선 근거로 기록)**.
- 전이 검증: 남(판교·분당) 학습 → 북(창릉·왕숙) 실측 예측 오차 정량화.

산출물: `data/processed/grid_features.geojson`(`lst_peak`, `lst_peak_filled` 추가)

In [1]:
# ── 실행 환경 가드 (능력 기반) ─────────────────────────────────
# 갭필 학습 재현에는 scikit-learn >= 1.4 가 필요합니다
# (HistGradientBoostingRegressor categorical_features="from_dtype", root_mean_squared_error).
# COMPAS(sklearn 1.1.3)에서는 아래 셀을 건너뛰고 동봉 산출물(lst_peak 포함
# grid_features.geojson)을 사용합니다 — 후속 노트북은 그대로 실행됩니다.
import pathlib
try:
    import sklearn
    _v = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    LOCAL_RUN = _v >= (1, 4)
except Exception:
    LOCAL_RUN = False
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if LOCAL_RUN:
    print("[로컬 모드] sklearn %s - 전체 셀을 실행합니다." % sklearn.__version__)
else:
    print("[COMPAS 모드] sklearn 버전 미충족(>=1.4 필요) -> 아래 셀은 건너뜁니다.")
    _p = _ROOT / "data/processed/grid_features.geojson"
    _r = _ROOT / "data/local/restore_lst_peak.csv.gz"
    if _p.exists():
        import geopandas as _gpd, pandas as _pd
        _g = _gpd.read_file(_p)
        if "lst_peak" in _g.columns and _g["lst_peak"].notna().any():
            print("  [OK] grid_features.geojson 에 lst_peak 존재 - 그대로 사용")
        elif _r.exists():
            _lp = _pd.read_csv(_r)
            _g = _g.drop(columns=[x for x in _lp.columns if x != "grid_id" and x in _g.columns], errors="ignore")
            _g = _g.merge(_lp, on="grid_id", how="left")
            _g.to_file(_p, driver="GeoJSON", COORDINATE_PRECISION=6)
            print("  [복원] restore_lst_peak.csv.gz -> grid_features.geojson 에 lst_peak 병합 (%d행)" % len(_lp))
        else:
            print("  [!!] lst_peak 없음 - data/local/restore_lst_peak.csv.gz 를 배치하십시오")
    else:
        print("  [!!] grid_features.geojson 없음 - 노트북 03 을 먼저 실행하십시오")


[로컬 모드] sklearn 1.6.1 - 전체 셀을 실행합니다.


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, geopandas as gpd
    from src.config import DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT, LANDSAT_PEAK, landsat_file
    from src.raster.landsat import lst_celsius
    from src.raster.geo import read_geotransform, pixel_center_coords

    g = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")
    # 멱등성: 이 노트북이 만드는 컬럼은 재실행 전 제거 (재읽기 충돌 방지)
    g = g.drop(columns=["lst_aug05", "lst_peak", "lst_peak_filled", "peak_lowconf"], errors="ignore")
    gL = g.to_crs(CRS_LANDSAT)
    g["cx"] = gL.geometry.centroid.x    # UTM 동향
    g["cy"] = gL.geometry.centroid.y    # UTM 북향(=위도 대리)

    # 8/5 피크를 격자에 집계 (타깃)
    aug = lst_celsius(LANDSAT_PEAK)
    gt = read_geotransform(str(landsat_file(LANDSAT_PEAK, "ST_B10")))
    xs, ys = pixel_center_coords(gt["gt"], gt["width"], gt["height"])
    v = ~np.isnan(aug)
    pts = gpd.GeoDataFrame({"a": aug[v]}, geometry=gpd.points_from_xy(xs[v], ys[v]), crs=CRS_LANDSAT)
    jn = gpd.sjoin(pts, gL[["grid_id", "geometry"]], predicate="within", how="inner")
    g = g.merge(jn.groupby("grid_id")["a"].mean().rename("lst_aug05"), on="grid_id", how="left")
    cov = g["lst_aug05"].notna().sum()
    print(f"8/5 실측 격자 {cov:,} ({100*cov/len(g):.0f}%)")

8/5 실측 격자 12,661 (23%)


In [3]:
if LOCAL_RUN:
    # 특징: 표면유형(ndvi, blockType) + 위치(cx, cy). 합성LST는 의도적 제외.
    g["blockType"] = g["blockType"].astype("category")
    FEAT = ["ndvi", "cx", "cy", "blockType"]
    train = g[g["lst_aug05"].notna() & g["ndvi"].notna()].copy()
    SOUTH = ["성남판교지구 택지개발사업", "성남분당"]
    NORTH = ["고양창릉 공공주택지구", "남양주왕숙 공공주택지구"]
    print("학습가능 격자:", len(train), "| 남", train.zoneName.isin(SOUTH).sum(), "| 북", train.zoneName.isin(NORTH).sum())

학습가능 격자: 12661 | 남 11567 | 북 1094


In [4]:
if LOCAL_RUN:
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.metrics import mean_absolute_error, root_mean_squared_error
    from sklearn.model_selection import train_test_split

    def hgb():
        return HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=42)

    # (A) 무작위 홀드아웃 — 기본 성능
    Xtr, Xte, ytr, yte = train_test_split(train[FEAT], train["lst_aug05"], test_size=0.2, random_state=42)
    p = hgb().fit(Xtr, ytr).predict(Xte)
    print(f"[무작위 홀드아웃] MAE {mean_absolute_error(yte,p):.2f}  RMSE {root_mean_squared_error(yte,p):.2f}")

    # (B) 공간 전이 — 남(판교·분당)만 학습 → 북(창릉·왕숙) 실측 예측
    s = train[train.zoneName.isin(SOUTH)]; n = train[train.zoneName.isin(NORTH)]
    pn = hgb().fit(s[FEAT], s["lst_aug05"]).predict(n[FEAT])
    print(f"[남→북 전이]    MAE {mean_absolute_error(n['lst_aug05'],pn):.2f}  RMSE {root_mean_squared_error(n['lst_aug05'],pn):.2f}  (실측평균 {n['lst_aug05'].mean():.1f})")

[무작위 홀드아웃] MAE 1.20  RMSE 1.63


[남→북 전이]    MAE 4.72  RMSE 6.02  (실측평균 34.7)


In [5]:
if LOCAL_RUN:
    # 전체 실측으로 학습 → 결측 격자 예측
    model = hgb().fit(train[FEAT], train["lst_aug05"])
    miss = g["lst_aug05"].isna() & g["ndvi"].notna()
    g["lst_peak"] = g["lst_aug05"].astype("float64")
    g.loc[miss, "lst_peak"] = model.predict(g.loc[miss, FEAT])
    g["lst_peak_filled"] = miss.astype("int8")   # 1=예측, 0=관측
    # 창릉·왕숙은 남→북 전이오차 큼 → 신뢰도 낮음 플래그
    g["peak_lowconf"] = (g["lst_peak_filled"] == 1) & g["zoneName"].isin(NORTH)
    print(f"lst_peak 커버리지 {g['lst_peak'].notna().sum():,} / {len(g):,}  (예측 {int(miss.sum()):,}, "
          f"저신뢰 북부예측 {int(g['peak_lowconf'].sum()):,})")
    print(g.groupby("zoneName")["lst_peak"].mean().round(1))

lst_peak 커버리지 54,029 / 54,029  (예측 41,368, 저신뢰 북부예측 20,811)
zoneName
고양창릉 공공주택지구      31.5
남양주왕숙 공공주택지구     35.3
성남분당             39.7
성남판교지구 택지개발사업    39.6
Name: lst_peak, dtype: float64


In [6]:
if LOCAL_RUN:
    g["blockType"] = g["blockType"].astype(str)
    g["peak_lowconf"] = g["peak_lowconf"].astype("int8")
    g.drop(columns=["cx", "cy"]).to_crs(CRS_SUBMIT).to_file(
        DATA_PROCESSED / "grid_features.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

    import matplotlib.pyplot as plt
    from src.viz.fonts import setup_font
    setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    g.plot(column="lst_peak", cmap="inferno", ax=axes[0], linewidth=0, legend=True,
           legend_kwds={"shrink": 0.6}, vmin=28, vmax=50)
    axes[0].set_title("8/5 피크 LST (갭필 후, full)")
    g.plot(column="lst_peak_filled", cmap="coolwarm", ax=axes[1], linewidth=0, legend=True,
           legend_kwds={"shrink": 0.6})
    axes[1].set_title("채움 여부 (1=예측, 0=관측)")
    for ax in axes:
        ax.set_axis_off()
    plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig06_lst_peak_gapfill.png", dpi=130); plt.close()